In [ ]:
import sys
import numpy as np

sys.path.append("..")
from src.qubo_windfarm_layout.model import load_wake_loss_data
from src.qubo_windfarm_layout.penalties import suggest_cardinality_penalty_from_layout, layout_yaml_to_z, suggest_spacing_penalty_from_layout, compute_lambda_from_lb, sdp_fixed_cardinality, sdp_spacing_violation

In [ ]:
GRID_RESOLUTION = 100
TIMEOUT = 3600
REFERENCE_LAYOUT = f"../results/layouts/cpsat_{200}_{TIMEOUT}s.yaml"

# 1. Problem Based Penalty Calibration

In [ ]:
lambda_cardinality_pb = (
    suggest_cardinality_penalty_from_layout(
        REFERENCE_LAYOUT
    )
)

lambda_spacing_pb = (
    suggest_spacing_penalty_from_layout(
        REFERENCE_LAYOUT
    )
)

print(f"Lambda cardinality = {lambda_cardinality_pb}")
print(f"Lambda cardinality = {lambda_spacing_pb}")

# 2. Relaxation-based penalty calibration 

In [7]:
from src.qubo_windfarm_layout.model import get_farm_area, get_mask
from src.solvers.utils import get_invalid_pairs
from src.qubo_windfarm_layout.evaluation import load_layout_coordinates

MIN_DISTANCE = 396  # 2 * 198 m

_, farm_area = get_farm_area()
X_grid, Y_grid, mask = get_mask(farm_area=farm_area, grid_resolution=GRID_RESOLUTION)
candidate_locations = np.column_stack([X_grid[mask], Y_grid[mask]])

invalid_pairs = get_invalid_pairs(
    candidate_locations=candidate_locations,
    min_distance=MIN_DISTANCE,
)

print(f"Candidati totali : {len(candidate_locations)}")
print(f"Coppie non valide: {len(invalid_pairs)}")

Candidati totali : 3622
Coppie non valide: 68686


In [8]:
# Step 1: compute upper bound using a feasible solution
z_reference, candidate_locations = layout_yaml_to_z(
    yaml_path=REFERENCE_LAYOUT,
    grid_resolution=GRID_RESOLUTION,
)

print(f"Candidati totali   : {len(candidate_locations)}")
print(f"Turbine selezionate: {z_reference.sum()}")

Candidati totali   : 3622
Turbine selezionate: 81


In [ ]:
# Step 2: compute lower bound using a relaxed solution
wake_loss = load_wake_loss_data(f"../results/precomputed/wake_loss_{GRID_RESOLUTION}m.npz")
wake_loss_matrix = wake_loss["wake_loss_matrix"]

LB_cardinality = sdp_fixed_cardinality(
    wake_loss_matrix,
    n_turbines=80,
    max_iters=1000
)

LB_spacing = sdp_spacing_violation(
    wake_loss_matrix,
    invalid_pairs,
    n_turbines=81,
    max_iters=1000
)

/home/maicolnicolini/code/qubo-wind-farm-layout/.venv/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:83: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
(CVXPY) Sep 15 03:02:09 PM: Your problem has 13122506 variables, 39374764 constraints, and 0 parameters.
(CVXPY) Sep 15 03:02:09 PM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Sep 15 03:02:09 PM: DCP verification time: 0.0002 seconds.
(CVXPY) Sep 15 03:02:09 PM: Expression tree has 10 nodes.
(CVXPY) Sep 15 03:02:09 PM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Sep 15 03:02:09 PM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) Sep 15 03:02:09 PM: Your problem is compiled with the CPP canonicalization backend.
(CVXPY) Sep 15 03:02:09 PM: Compiling problem (target solver=SCS).
(CVXPY) Sep 15 03:02:09 PM: Reduction chai

                                     CVXPY                                     
                                     v1.9.2                                    
-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------


(CVXPY) Sep 15 03:02:09 PM: Applying reduction ExactCone2Cone
(CVXPY) Sep 15 03:02:11 PM: Applying reduction EliminateZeroSized
(CVXPY) Sep 15 03:02:11 PM: Applying reduction ConeMatrixStuffing
(CVXPY) Sep 15 03:03:05 PM: Applying reduction SCS
(CVXPY) Sep 15 03:03:19 PM: Finished problem compilation (took 7.048e+01 seconds).
(CVXPY) Sep 15 03:03:19 PM: Invoking solver SCS  to obtain a solution.


-------------------------------------------------------------------------------
                                Numerical solver                               
-------------------------------------------------------------------------------
------------------------------------------------------------------
	       SCS v3.3.1 - Splitting Conic Solver
	(c) Brendan O'Donoghue, Stanford University, 2012
------------------------------------------------------------------
problem:  variables n: 6564875, constraints m: 32813511
cones: 	  z: primal zero / dual free vars: 3623
	  l: linear vars: 26245012
	  s: psd vars: 6564876, ssize: 1
settings: eps_abs: 1.0e-04, eps_rel: 1.0e-04, eps_infeas: 1.0e-07
	  alpha: 1.50, scale: 1.00e-01, adaptive_scale: 1
	  max_iters: 1000, normalize: 1, rho_x: 1.00e-06
	  acceleration_lookback: 10, acceleration_interval: 5
lin-sys:  sparse-indirect-scs
	  nnz(A): 32820753, nnz(P): 0
------------------------------------------------------------------
 iter | pri res

In [ ]:
# Step 3: compute lambda
z_reference = layout_yaml_to_z(yaml_path=REFERENCE_LAYOUT, grid_resolution=GRID_RESOLUTION)[0]

lambda_cardinality = compute_lambda_from_lb(wake_loss_matrix=wake_loss_matrix, z_reference=z_reference, lower_bound=LB_cardinality)
lambda_spacing = compute_lambda_from_lb(wake_loss_matrix=wake_loss_matrix, z_reference=z_reference, lower_bound=LB_spacing)

In [ ]:
print(f"Lambda cardinality = {lambda_cardinality}")
print(f"Lambda cardinality = {lambda_spacing}")